# Mini-Project: Data Analysis for Marketing Strategy

This notebook analyzes the US Superstore dataset to identify high-value states, cities, customers, and product drivers for marketing strategy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
data_path = '../Day_4/Sample - Superstore.csv'
df = pd.read_csv(data_path, encoding='latin1', parse_dates=['Order Date', 'Ship Date'])
df.head()

In [ ]:
df.shape, df.columns.tolist(), df.isna().sum()

## Preprocessing
- Parse dates
- Confirm no missing values in key columns
- Create useful summary features for analysis.

In [ ]:
df = df.drop_duplicates()
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.to_period('M')
df['Profit Margin'] = df['Profit'] / df['Sales'].replace(0, np.nan)
df[['Order Date', 'Sales', 'Profit', 'State', 'City']].head()

## 1. Which states have the most sales?

In [ ]:
state_sales = df.groupby('State', observed=True)['Sales'].sum().sort_values(ascending=False).reset_index()
state_sales.head(15)

In [ ]:
top_states = state_sales.head(10)
sns.barplot(data=top_states, x='Sales', y='State', palette='crest')
plt.title('Top 10 States by Total Sales')
plt.xlabel('Total Sales')
plt.ylabel('State')
plt.tight_layout()

## 2. Compare sales and profit for New York vs California

In [ ]:
state_compare = df[df['State'].isin(['New York', 'California'])].groupby('State', observed=True)[['Sales', 'Profit']].sum().reset_index()
state_compare

## 3. Outstanding customers in New York

In [ ]:
ny_customers = df[df['State'] == 'New York'].groupby(['Customer ID', 'Customer Name'], observed=True)[['Sales', 'Profit']].sum().reset_index()
ny_customers.sort_values(by='Profit', ascending=False).head(10)

## 4. State profitability differences

In [ ]:
state_profitability = df.groupby('State', observed=True).agg({'Sales':'sum','Profit':'sum'}).reset_index()
state_profitability['Profit Rate'] = state_profitability['Profit'] / state_profitability['Sales']
state_profitability.sort_values(by='Profit Rate', ascending=False).head(15)

In [ ]:
top_profit_states = state_profitability[state_profitability['Sales'] > state_profitability['Sales'].quantile(0.5)]
sns.scatterplot(data=top_profit_states, x='Sales', y='Profit Rate', hue='State', s=120, legend=False)
plt.title('Profitability Rate vs Sales for Higher-Sales States')
plt.xlabel('Total Sales')
plt.ylabel('Profit Rate')
plt.tight_layout()

## 5. Pareto Principle for customers and Profit
Determine whether 20% of customers contribute roughly 80% of profit.

In [ ]:
customer_profit = df.groupby(['Customer ID', 'Customer Name'], observed=True)['Profit'].sum().reset_index()
customer_profit = customer_profit.sort_values(by='Profit', ascending=False).reset_index(drop=True)
customer_profit['CumProfit'] = customer_profit['Profit'].cumsum()
customer_profit['CumProfitPct'] = customer_profit['CumProfit'] / customer_profit['Profit'].sum()
customer_profit['CumCountPct'] = (customer_profit.index + 1) / len(customer_profit)
customer_profit.head(10)
num_top_20 = int(np.ceil(len(customer_profit) * 0.2))
top20_profit_pct = customer_profit.loc[:num_top_20 - 1, 'Profit'].sum() / customer_profit['Profit'].sum()
num_top_20, top20_profit_pct

## 6. Top 20 cities by Sales and by Profit
Analyze whether the leading cities by sales are also the most profitable.

In [ ]:
city_sales = df.groupby('City', observed=True)[['Sales', 'Profit']].sum().reset_index()
top20_sales_cities = city_sales.sort_values(by='Sales', ascending=False).head(20)
top20_profit_cities = city_sales.sort_values(by='Profit', ascending=False).head(20)
top20_sales_cities[['City', 'Sales', 'Profit']].reset_index(drop=True)
top20_profit_cities[['City', 'Sales', 'Profit']].reset_index(drop=True)

In [ ]:
plt.figure(figsize=(14, 7))
sns.barplot(data=top20_sales_cities, x='Sales', y='City', palette='mako')
plt.title('Top 20 Cities by Total Sales')
plt.xlabel('Total Sales')
plt.ylabel('City')
plt.tight_layout()
plt.show()
plt.figure(figsize=(14, 7))
sns.barplot(data=top20_profit_cities, x='Profit', y='City', palette='rocket')
plt.title('Top 20 Cities by Total Profit')
plt.xlabel('Total Profit')
plt.ylabel('City')
plt.tight_layout()

## 7. Top 20 customers by Sales


In [ ]:
top20_sales_customers = df.groupby(['Customer ID', 'Customer Name'], observed=True)['Sales'].sum().reset_index().sort_values(by='Sales', ascending=False).head(20)
top20_sales_customers

## 8. Cumulative Sales Curve by Customers and Pareto assessment
Plot the sales concentration and check if 20% of customers drive ~80% of sales.

In [ ]:
customer_sales = df.groupby(['Customer ID', 'Customer Name'], observed=True)['Sales'].sum().reset_index()
customer_sales = customer_sales.sort_values(by='Sales', ascending=False).reset_index(drop=True)
customer_sales['CumSales'] = customer_sales['Sales'].cumsum()
customer_sales['CumSalesPct'] = customer_sales['CumSales'] / customer_sales['Sales'].sum()
customer_sales['CustomerPct'] = (customer_sales.index + 1) / len(customer_sales)
plt.figure(figsize=(12, 6))
plt.plot(customer_sales['CustomerPct'], customer_sales['CumSalesPct'], marker='o', linewidth=2)
plt.axhline(0.8, color='red', linestyle='--', label='80% Sales')
plt.axvline(0.2, color='green', linestyle='--', label='20% Customers')
plt.title('Cumulative Sales Percentage by Customer Rank')
plt.xlabel('Share of Customers')
plt.ylabel('Cumulative Share of Sales')
plt.legend()
plt.tight_layout()
num_top_20_sales = int(np.ceil(len(customer_sales) * 0.2))
top20_sales_pct = customer_sales.loc[:num_top_20_sales - 1, 'Sales'].sum() / customer_sales['Sales'].sum()
num_top_20_sales, top20_sales_pct

## 9. Marketing strategy recommendations
Based on the analysis, prioritize states, cities, and top customers for targeted campaigns.

In [ ]:
priority_states = state_sales.head(5)['State'].tolist()
priority_cities = top20_sales_cities['City'].tolist()
priority_states, priority_cities[:10]